# E1 — CNN 1D con contexto temporal por concatenación

**Proyecto:** Efecto del contexto temporal en la clasificación de estadios del sueño  
**Grupo 25 — Universidad de los Andes, 2026**  
**Responsable:** Omar Díaz Hernández

---

Este notebook entrena el experimento E1: CNN 1D con ventana de 5 épocas consecutivas (contexto por concatenación).  
Comparte el extractor convolucional con E0 (misma arquitectura, pesos no transferidos — se entrena desde cero).  
Los hiperparámetros son idénticos a E0 (`config.py`), calibrados con la corrida exploratoria de E0.

**Protocolo:** 3 corridas con semillas 42, 123 y 2026.

## 0. Entorno y dependencias

In [ ]:
# Verificar GPU disponible
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Instalar dependencias faltantes en Colab
%pip install -q mlflow huggingface_hub scikit-learn

## 1. Clonar el repositorio y situarse en la rama correcta

In [ ]:
import os

REPO_URL    = "https://github.com/amquinteroc1/maia-grupo25-proyecto.git"
REPO_BRANCH = "training/e0-e1-e2"
REPO_DIR    = "/content/maia-grupo25-proyecto"

if not os.path.exists(REPO_DIR):
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!git log --oneline -3
print(f"\nDirectorio de trabajo: {os.getcwd()}")

In [ ]:
# Agregar la raíz del repo al path para importar src.*
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verificar imports
from src.config import PATHS, CNN_ARCH, E1_HEAD, TRAINING, SEEDS, SEED_LABELS, N_CLASSES, CLASS_NAMES, MLFLOW
print("✓ src.config importado")
print(f"  Semillas: {SEEDS}")
print(f"  max_epochs: {TRAINING['max_epochs']}")
print(f"  early_stopping_patience: {TRAINING['early_stopping_patience']}")
print(f"  seq_len (E1): {E1_HEAD['seq_len']}")
print(f"  in_features cabezal: {E1_HEAD['in_features']}  (5 × 512)")

## 2. Descargar datos desde HuggingFace

> Los datos ya están normalizados (Diego). **No volver a normalizar.**

In [ ]:
# Login a HuggingFace (repo privado)
from huggingface_hub import login
login()   # abre un prompt para pegar el token HF

In [ ]:
from huggingface_hub import snapshot_download

HF_REPO    = "maia-grupo25/sleep-edfx-data"
LOCAL_DATA = REPO_DIR + "/data/processed"

os.makedirs(LOCAL_DATA, exist_ok=True)

snapshot_download(
    repo_id   = HF_REPO,
    repo_type = "dataset",
    local_dir = LOCAL_DATA,
)
print("\nArchivos descargados:")
!find {LOCAL_DATA} -name "*.npy" | sort | head -30

## 3. Verificar estructura de datos

In [ ]:
import numpy as np

data_seq = PATHS["data_seq"]

print("Dataset secuencial (E1/E2):")
for split in ["train", "val", "test", "st"]:
    y = np.load(data_seq / f"y_seq_{split}.npy")
    g = np.load(data_seq / f"groups_seq_{split}.npy")
    print(f"  {split:5s}: {len(y):>8,} secuencias | {len(np.unique(g))} sujetos")

# Verificar shape de X_seq
X_sample = np.load(str(data_seq / "X_seq_train.npy"), mmap_mode="r")
print(f"\nShape X_seq_train: {X_sample.shape}  ← esperado (N, 5, 3, 3000)")

## 4. Configurar MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri(MLFLOW["tracking_uri"])
mlflow.set_experiment(MLFLOW["experiment_name"])
print(f"MLflow tracking URI: {MLFLOW['tracking_uri']}")
print(f"Experimento        : {MLFLOW['experiment_name']}")

## 5. Funciones auxiliares

In [ ]:
import random
import torch

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"  Semilla fijada: {seed}")

def get_device() -> torch.device:
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    else:
        dev = torch.device("cpu")
        print("  CPU (no GPU disponible)")
    return dev

## 6. Corridas oficiales E1 — 3 semillas

> Los hiperparámetros ya están calibrados desde la corrida exploratoria de E0.  
> `max_epochs` y `early_stopping_patience` se toman directamente de `config.py`.

In [ ]:
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

from src.models.cnn1d_e1 import SleepCNN1D_E1
from src.training.dataset import SleepDatasetSeq, compute_class_weights
from src.training.trainer import run_training

print("✓ Imports E1 correctos")

# Verificación rápida del modelo
dummy_model = SleepCNN1D_E1()
dummy_input = torch.randn(2, 5, 3, 3000)
dummy_out   = dummy_model(dummy_input)
assert dummy_out.shape == (2, 5), f"Shape incorrecto: {dummy_out.shape}"
n_params = sum(p.numel() for p in dummy_model.parameters() if p.requires_grad)
print(f"✓ Modelo E1 verificado — {n_params:,} parámetros")
del dummy_model, dummy_input, dummy_out

In [ ]:
all_results = {}

for seed in SEEDS:
    label = SEED_LABELS[seed]
    print(f"\n{'='*55}")
    print(f"  E1 — Semilla {seed} (Condición {label})")
    print(f"{'='*55}")

    set_seed(seed)
    device = get_device()

    # Datasets secuenciales — SleepDatasetSeq ya trabaja con la cohorte común
    train_ds = SleepDatasetSeq(split="train")
    val_ds   = SleepDatasetSeq(split="val")
    test_ds  = SleepDatasetSeq(split="test")
    st_ds    = SleepDatasetSeq(split="st")

    print(f"\nTamaños:")
    print(f"  SC-train : {len(train_ds):,}")
    print(f"  SC-val   : {len(val_ds):,}")
    print(f"  SC-test  : {len(test_ds):,}")
    print(f"  ST       : {len(st_ds):,}")

    pin = TRAINING["pin_memory"] and device.type == "cuda"
    bs  = TRAINING["batch_size"]

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=0, pin_memory=pin,
                              worker_init_fn=lambda wid: np.random.seed(seed + wid))
    val_loader   = DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)
    test_loader  = DataLoader(test_ds,  batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)
    st_loader    = DataLoader(st_ds,    batch_size=bs, shuffle=False, num_workers=0, pin_memory=pin)

    class_weights = compute_class_weights(train_ds.get_labels(), n_classes=N_CLASSES).to(device)
    print(f"\nPesos: { {n: round(float(w), 3) for n, w in zip(CLASS_NAMES, class_weights.cpu())} }")

    model = SleepCNN1D_E1(
        n_channels=CNN_ARCH["n_channels"],
        n_classes=N_CLASSES,
        dropout=CNN_ARCH["dropout"],
        seq_len=E1_HEAD["seq_len"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=TRAINING["lr"], weight_decay=TRAINING["weight_decay"])
    scheduler = ReduceLROnPlateau(optimizer, mode="max",
                                   patience=TRAINING["scheduler_patience"],
                                   factor=TRAINING["scheduler_factor"],
                                   min_lr=TRAINING["scheduler_min_lr"])
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    result = run_training(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        experiment="E1",
        seed=seed,
        run_dir=PATHS["experiments"] / "E1" / f"seed{seed}",
        groups_test=test_ds.get_groups(),
        st_loader=st_loader,
        groups_st=st_ds.get_groups(),
    )
    all_results[seed] = result

## 7. Resumen final E1

In [ ]:
print("\n" + "="*55)
print("RESUMEN E1 — 3 corridas")
print("="*55)

f1s = [r["test_metrics"]["test_f1_macro"] for r in all_results.values()]
n1s = [r["test_metrics"]["test_f1_n1"]    for r in all_results.values()]
kas = [r["test_metrics"]["test_kappa"]     for r in all_results.values()]
bas = [r["test_metrics"]["test_balanced_accuracy"] for r in all_results.values()]

print(f"  test_f1_macro     : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
print(f"  test_f1_N1        : {np.mean(n1s):.4f} ± {np.std(n1s):.4f}")
print(f"  test_kappa        : {np.mean(kas):.4f} ± {np.std(kas):.4f}")
print(f"  test_balanced_acc : {np.mean(bas):.4f} ± {np.std(bas):.4f}")
print()

for seed, r in all_results.items():
    lbl = SEED_LABELS[seed]
    m   = r["test_metrics"]
    print(
        f"  [{lbl}] seed={seed} | "
        f"f1_macro={m['test_f1_macro']:.4f} | "
        f"f1_N1={m['test_f1_n1']:.4f} | "
        f"kappa={m['test_kappa']:.4f} | "
        f"ep={r['best_epoch']}"
    )

print("\nArtefactos para Luigi guardados en:")
for seed in SEEDS:
    print(f"  experiments/E1/seed{seed}/")

## 8. Subir artefactos a HuggingFace

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path=str(PATHS["experiments"] / "E1"),
    path_in_repo="experiments/E1",
    repo_id="maia-grupo25/sleep-edfx-data",
    repo_type="dataset",
    create_pr=True,
)
print("✓ PR creado con todos los artefactos de E1")